# Silver: Product — Bronze → Silver Cleansing

Operations performed:
- Deduplicate
- Trim / case normalisation
- Type casting
- Incremental load via `ingest_ts` watermark

In [0]:
dbutils.widgets.dropdown(name="environment", defaultValue="dev",
                         choices=["dev","qa","prd"], label="select Environment")
env = dbutils.widgets.get("environment")


In [0]:
silverTablName = f"saleslake_{env}.silver_{env}.cleanedProduct"
bronzeTablName = f"saleslake_{env}.bronze_{env}.rawProduct"
print(f"Source : {bronzeTablName}")
print(f"Target : {silverTablName}")


In [0]:
spark.sql(f"""
INSERT INTO {silverTablName}
SELECT DISTINCT
    CAST(TRIM(product_id) AS INT)             AS product_id,
    UPPER(TRIM(sku))                          AS sku,
    UPPER(TRIM(product_name))                       AS product_name,
    UPPER(TRIM(category))                     AS category,
    UPPER(TRIM(sub_category))                 AS sub_category,
    UPPER(TRIM(brand))                        AS brand,
    UPPER(TRIM(supplier))                     AS supplier,
    CAST(TRIM(unit_cost) AS DECIMAL(18,2))    AS unit_cost,
    CAST(TRIM(list_price) AS DECIMAL(18,2))   AS list_price,
    TO_DATE(TRIM(launch_date), 'yyyy-MM-dd')  AS launch_date,
    UPPER(TRIM(status))                       AS status,
    CURRENT_TIMESTAMP() AS ingest_ts
FROM {bronzeTablName}
WHERE ingest_ts > (
    SELECT COALESCE(MAX(ingest_ts), TO_TIMESTAMP("1990-01-01"))
    FROM {silverTablName}
)
ORDER BY product_id
""")

print(f"Silver load complete for {silverTablName}")
spark.sql(f"SELECT COUNT(*) AS row_count FROM {silverTablName}").display()
